# 06 · Integración gold
**Responsable:** P6 · **Código:** `src/gold_integracion.py` · **Salidas:** `gold/municipio_anio.parquet`, `gold/maestro_boyaca.geoparquet`

**Qué hace**
- Parte de la población (123 municipios × 6 años) y le une con LEFT JOIN: el desempeño del bloque oficial (solo 2020), los totales agrícolas (solo 2023), el % rural y el resumen de suelo.
- Cada fuente llega ya agregada a su grano, así que una unión no puede duplicar filas. Si el número de filas cambia o la población total cambia, el pipeline se **detiene**.
- En cada unión registra cuántas claves coinciden y guarda las no vinculadas en `outputs/quality/no_vinculados_<fuente>.csv`.
- Calcula los indicadores: las 4 proporciones demográficas y la producción por cada 1.000 habitantes de 18–64 años. Una división entre 0 da null.
- `maestro_boyaca`: una fila por municipio con las variables de 2020 y de 2023 más la geometría, para los mapas.

**Por qué años distintos:** el desempeño solo existe para 2020 y la producción solo para 2023. Cada variable vive en su año y en los demás queda null, nunca 0.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import polars as pl
from src.config import *
asegurar_carpetas()

In [2]:
import geopandas as gpd
from src.gold_agricola import resumen_municipal
from src.gold_integracion import construir_maestro, construir_municipio_anio
from src.silver_poblacion import pct_rural

pob = pl.read_parquet(SILVER / "poblacion.parquet")
areas = pl.read_parquet(SILVER / "poblacion_area.parquet")
mdm = pl.read_parquet(SILVER / "desempeno.parquet")
cm = pl.read_parquet(GOLD / "cultivo_municipio.parquet")
suelo_res = pl.read_parquet(GOLD / "suelo_resumen.parquet")
mun = gpd.read_parquet(SILVER / "municipios.geoparquet")

In [3]:
ma = construir_municipio_anio(pob, mdm, resumen_municipal(cm), pct_rural(areas), suelo_res)
ma.group_by("anio").agg(pl.len(), pl.col("puntaje_mdm").count(), pl.col("produccion_total_t").count()).sort("anio")

[gold/municipio_anio] revisados=738 validos=738 rechazados=0


anio,len,puntaje_mdm,produccion_total_t
i16,u32,u32,u32
2020,123,123,0
2021,123,0,0
2022,123,0,0
2023,123,0,123
2024,123,0,0
2025,123,0,0


In [4]:
maestro = construir_maestro(ma, mun)
maestro.head()

,codigo_municipio,municipio,geometry,prop_18_44_pct_2020,prop_mujeres_18_44_pct_2020,prop_hombres_18_44_pct_2020,pct_rural_2020,grupo_capacidad_2020,puntaje_mdm_2020,poblacion_total_2020,pob_18_64_2023,prop_18_64_pct_2023,produccion_total_t_2023,produccion_papa_t_2023,rendimiento_papa_t_ha_2023,produccion_1k_18_64_2023,clase_predominante,pct_sin_clasificar,pct_clases_iii_iv
0,15001,Tunja,"POLYGON ((-73.41186 5.56682, -73.41006 5.56651...",43.851376,22.779159,21.072217,4.352557,G1,66.23,178975,121900,65.626548,106100.347559,90000.0,18.987342,870.388413,IV,6.254662e-13,33.025012
1,15022,Almeida,"POLYGON ((-73.37179 5.01386, -73.36445 5.01426...",27.316040,11.911064,15.404976,81.605691,G4,42.41,1889,1003,54.333694,4337.150000,55.8,7.971429,4324.177468,VII,8.067652e-13,0.000000
2,15047,Aquitania,"POLYGON ((-72.76346 5.64502, -72.76286 5.63893...",37.332561,18.199382,19.133179,61.312761,G2,51.80,15528,9106,58.938511,87274.089518,1382.0,16.163743,9584.240009,VIII,2.321285e+00,4.137757
3,15051,Arcabuco,"POLYGON ((-73.47156 5.8216, -73.47348 5.81683,...",38.388158,19.210526,19.177632,58.523668,G2,52.73,6080,3785,60.155753,66302.743592,64375.0,25.000000,17517.237409,VI,1.090882e+00,11.886260
4,15087,Belén,"POLYGON ((-72.81735 6.01592, -72.81445 6.02122...",36.183086,18.321715,17.861371,38.544650,G2,49.16,7603,4446,58.277625,13083.744979,10950.0,17.244094,2942.812636,VIII,1.271761e+00,13.658694


In [5]:
from src import graficos
graficos.distribucion_rural(ma.to_pandas(), ANIO_MDM)        # P1
graficos.produccion_por_1000(ma.to_pandas(), ANIO_EVA)        # P2

PosixPath('/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/proyecto_boyaca/outputs/figures/p2_produccion_por_1000_2023.png')

In [6]:
pl.read_csv(QUALITY / "metricas_gold.csv")

fecha_ejecucion,capa,fuente,regla,descripcion,revisados,afectados,resultado,accion
str,str,str,str,str,i64,i64,str,str
"""2026-11-05T12:51:12""","""gold""","""cultivo_municipio""","""rendimiento_division_por_cero""","""Cultivos con área cosechada 0:…",1731,56,"""FALLA""","""alertar"""
"""2026-11-05T12:51:12""","""gold""","""cultivo_municipio""","""rendimiento_finito""","""Sin infinitos ni NaN""",1731,0,"""OK""","""detener"""
"""2026-11-05T12:51:12""","""gold""","""cultivo_municipio""","""rendimiento_no_negativo""","""Rendimiento >= 0""",1731,0,"""OK""","""detener"""
"""2026-11-05T12:51:12""","""gold""","""cultivo_municipio""","""conciliacion_produccion_silver""","""Producción total gold - silver…",2613,0,"""OK""","""alertar"""
"""2026-11-05T12:52:01""","""gold""","""suelo""","""municipios_sin_cobertura""","""Municipios que la capa IGAC no…",123,0,"""OK""","""alertar"""
…,…,…,…,…,…,…,…,…
"""2026-11-05T12:53:22""","""gold""","""municipio_anio""","""cardinalidad_anio_2024""","""anio=2024: se esperaban 123 mu…",123,0,"""OK""","""alertar"""
"""2026-11-05T12:53:22""","""gold""","""municipio_anio""","""cardinalidad_anio_2020""","""anio=2020: se esperaban 123 mu…",123,0,"""OK""","""alertar"""
"""2026-11-05T12:53:22""","""gold""","""municipio_anio""","""cardinalidad_anio_2023""","""anio=2023: se esperaban 123 mu…",123,0,"""OK""","""alertar"""
